In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix
from sklearn.impute import SimpleImputer
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
file_path = 'data/raw/datafile_full.csv' 
df = pd.read_csv(file_path)

if 'ID_code' in df.columns:
    df = df.drop('ID_code', axis=1)

print(f"Data Shape: {df.shape}")
print(f"Total NA Values: {df.isnull().sum().sum()}")
print(df.describe().iloc[:, :5])

X = df.drop('target', axis=1)
y = df['target']

In [ ]:
plt.figure(figsize=(8, 6))


target_counts = df['target'].value_counts()

ax = sns.barplot(
    x=target_counts.index, 
    y=target_counts.values, 
    hue=target_counts.index,  
    palette=['#1f77b4', '#d62728'], 
    legend=False )

plt.title('Target Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Target Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks([0, 1], ['No Application (0)', 'Application (1)'])


total = len(df)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        percentage = f'{100 * height / total:.1f}%'
        ax.annotate(f'{int(height):,}\n({percentage})', 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', 
                    xytext=(0, 5), 
                    textcoords='offset points',
                    fontsize=12, fontweight='bold', color='black')


plt.tight_layout()
plt.savefig('Figure1_Class_Distribution.png', dpi=300)
plt.show()

In [ ]:
missing_mask = df.isnull().any(axis=1)
missing_counts = df[missing_mask]['target'].value_counts().sort_index()
total_counts = df['target'].value_counts().sort_index()

plot_df = pd.DataFrame({
    'Target Class': ['No Application (0)', 'Application (1)'],
    'Total Customers': total_counts.values,
    'Incomplete Records (NA)': missing_counts.values})

plot_df_melted = plot_df.melt(id_vars='Target Class', var_name='Record Type', value_name='Count')


plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Target Class', y='Count', hue='Record Type', 
                 data=plot_df_melted, palette=['#e0e0e0', '#c44e52']) 

plt.title('Missing Values Distribution by Class', fontsize=14, fontweight='bold')
plt.xlabel('Customer Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(title=None)


for i, p in enumerate(ax.patches):
    height = p.get_height()
    if height > 0:
      
        if i >= 2: 
            class_idx = i - 2
            total_ref = total_counts.values[class_idx]
            pct = 100 * height / total_ref
            label = f'{int(height):,}\n({pct:.1f}%)'
        else:
            label = f'{int(height):,}'
            
        ax.annotate(label, 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', 
                    fontsize=11, fontweight='bold', color='black')

plt.tight_layout()
plt.savefig('Figure2_Missing_Values.png', dpi=300)
plt.show()

In [ ]:

# We use Z-Score > 4 to remove extreme anomalies. 

train_mean = X.mean()
train_std = X.std()

z_scores = (X - train_mean) / train_std
# Filling NaNs with 0 (Mean) so they aren't counted as outliers
z_scores = z_scores.fillna(0) 

mask = (z_scores.abs() < 4).all(axis=1)

X_clean = X[mask]
y_clean = y[mask]

removed = len(X) - len(X_clean)
print(f"Outliers Removed: {removed} rows")
print(f"New Data Shape:   {X_clean.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean)

In [ ]:
# Fit on Train, Transform Train & Test (No Leakage)
imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train)

cols = X_train.columns
X_train = pd.DataFrame(imputer.transform(X_train), columns=cols)
X_test = pd.DataFrame(imputer.transform(X_test), columns=cols)
print("Imputation Complete.")

In [ ]:
scout = xgb.XGBClassifier(
    n_estimators=50, max_depth=4, n_jobs=-1, random_state=42)
scout.fit(X_train, y_train)

importance = scout.get_booster().get_score(importance_type='gain')
all_features = set(X_train.columns)
used_features = set(importance.keys())

low_gain_features = [k for k, v in importance.items() if v <= 0.05]
unused_features = list(all_features - used_features)
drop_list = low_gain_features + unused_features

print(f"Total Features: {len(all_features)}")
print(f"Dropping {len(drop_list)} features (Gain <= 0.05).")

In [ ]:
X_train_pruned = X_train.drop(columns=drop_list)
X_test_pruned = X_test.drop(columns=drop_list)
print(f"Features Remaining: {X_train_pruned.shape[1]}")

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train_pruned, y_train)

print(f"Train Size Before: {X_train_pruned.shape}")
print(f"Train Size After:  {X_train_bal.shape}")
print(f"Class Distribution:\n{y_train_bal.value_counts()}")

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.5,
    gamma=1, 
    reg_alpha=10, 
    reg_lambda=10,
    n_jobs=-1, 
    random_state=42, 
    tree_method='hist',
    early_stopping_rounds=100)

model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test_pruned, y_test)],
    verbose=0)

In [ ]:
plt.figure(figsize=(12, 10))

lgb.plot_importance(
    model_lgb, 
    max_num_features=20, 
    importance_type='gain', 
    figsize=(12, 8),
    height=0.7,
    title='LightGBM Feature Importance (Top 20 by Gain)',
    xlabel='Feature Importance (Gain)',
    grid=False)

plt.tight_layout()
plt.show()

In [ ]:
model_lgb = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=31,           
    max_bin=255,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary',
    metric='auc',
    n_jobs=-1,
    random_state=42,
    verbosity=-1)

callbacks = [lgb.early_stopping(100), lgb.log_evaluation(0)]

model_lgb.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test_pruned, y_test)],
    eval_metric='auc',
    callbacks=callbacks)

y_prob_lgb = model_lgb.predict_proba(X_test_pruned)[:, 1]
auc_lgb = roc_auc_score(y_test, y_prob_lgb)
print(f"LightGBM AUC Score: {auc_lgb:.4f}")

In [ ]:
y_prob_lgb = model_lgb.predict_proba(X_test_pruned)[:, 1]

auc_lgb = roc_auc_score(y_test, y_prob_lgb)
print(f"LightGBM AUC: {auc_lgb:.4f}")

p, r, thresholds = precision_recall_curve(y_test, y_prob_lgb)
f1_scores = 2 * (p * r) / (p + r + 1e-9)
best_idx = np.argmax(f1_scores)
best_thresh_lgb = thresholds[best_idx]

print(f"Best Threshold (LGBM): {best_thresh_lgb:.4f}")

y_pred_lgb = (y_prob_lgb >= best_thresh_lgb).astype(int)


print(classification_report(y_test, y_pred_lgb))

cm_lgb = confusion_matrix(y_test, y_pred_lgb)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_lgb, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])
plt.title(f"LightGBM Confusion Matrix (Thresh={best_thresh_lgb:.2f})")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

In [ ]:
y_prob = model.predict_proba(X_test_pruned)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"Final AUC Score: {auc:.4f}")

p, r, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_pred = (y_prob >= best_threshold).astype(int)

print(f"Best Threshold (Max F1): {best_threshold:.4f}")
print(classification_report(y_test, y_pred, digits=4))


plt.figure(figsize=(12, 5))


plt.subplot(1, 2, 1)
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f"XGBoost (AUC={auc:.3f})", lw=2)
plt.plot([0, 1], [0, 1], 'k--')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()


plt.subplot(1, 2, 2)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f"Confusion Matrix (Thresh={best_threshold:.2f})")
plt.ylabel("Actual")
plt.xlabel("Predicted")

plt.tight_layout()
plt.show()

In [ ]:
print(f"XGBoost AUC:  {auc:.4f}")
print(f"LightGBM AUC: {auc_lgb:.4f}")

plt.figure(figsize=(10, 6))

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob)
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={auc:.3f})", color='blue', lw=2)

fpr_lgb, tpr_lgb, _ = roc_curve(y_test, y_prob_lgb)
plt.plot(fpr_lgb, tpr_lgb, label=f"LightGBM (AUC={auc_lgb:.3f})", color='orange', lw=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title("Model Comparison: ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()